# BÀI TẬP THỰC HÀNH CHƯƠNG 5
## Thị giác máy tính (Computer Vision)

---

## 📁 Cấu trúc thư mục

```
chapter_5_lab/
├── images/              # Ảnh đầu vào
├── output/              # Kết quả xử lý
└── chapter_5_lab.ipynb  # File notebook (đặt trực tiếp ở thư mục ngoài)
```

---

## ⚙️ 0. Chuẩn bị môi trường

**Cell 0 — Khởi tạo chung (chạy đầu tiên trong notebook):**

In [ ]:
%matplotlib inline

import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from skimage import data, measure, color, feature
from skimage.transform import rotate
from collections import Counter

Path("images").mkdir(exist_ok=True)
Path("output").mkdir(exist_ok=True)

print("OpenCV version:", cv2.__version__)
print("NumPy version :", np.__version__)
print("Haar cascade folder:", cv2.data.haarcascades)

import urllib.request
import os

# Danh sách các file XML cần tải
xml_files = {
    "haarcascade_frontalface_default.xml": "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml",
    "haarcascade_eye.xml": "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_eye.xml",
    "haarcascade_smile.xml": "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_smile.xml"
}

print("Đang tải các file Haar Cascade XML...")
for filename, url in xml_files.items():
    filepath = os.path.join("images", filename)
    if not os.path.exists(filepath):
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"  ✓ Đã tải: {filename}")
        except Exception as e:
            print(f"  ✗ Lỗi tải {filename}: {e}")
    else:
        print(f"  - Đã có sẵn: {filename}")
print("Hoàn tất!")

# ---------- Hàm tiện ích hiển thị ----------
def show_image(img, title="", cmap=None, figsize=(5, 4), save_path=None):
    plt.figure(figsize=figsize)
    if img.ndim == 3:
        plt.imshow(img)
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title)
    plt.axis('off')
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()


def show_grid(images, titles, ncols=3, figsize=(15, 8),
              cmap='gray', save_path=None, main_title=None):
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).ravel()
    for i, (im, t) in enumerate(zip(images, titles)):
        ax = axes[i]
        if im.ndim == 3:
            ax.imshow(im)
        else:
            ax.imshow(im, cmap=cmap)
        ax.set_title(t, fontsize=10)
        ax.axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')
    if main_title:
        plt.suptitle(main_title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()

> **Lưu ý:** Không dùng `cv2.imshow` / `cv2.waitKey` trong notebook.

---

## Bài 1: Đặc trưng ảnh đơn giản — Histogram màu

**Mức độ:** Cơ bản

**Mục tiêu:** Hiểu cách xây dựng **feature vector** từ ảnh (bước đệm cho phân loại ảnh).

**Yêu cầu:**
1. Tải ảnh `data.astronaut()`.
2. Cắt ra 3 vùng: mặt người, nền xanh (sky), bộ đồ (clothing).
3. Tính **histogram màu 3D** (kênh R, G, B) cho mỗi vùng, chuyển về vector 1D.
4. Vẽ histogram của từng vùng.
5. So sánh: dùng `np.linalg.norm` để tính khoảng cách giữa các feature vector.
6. Lưu kết quả vào `output/bai1_color_histogram.png`.

**Lời giải:**

In [ ]:
img_rgb = data.astronaut()

# 1. Cắt 3 vùng
face     = img_rgb[0:180,  150:300]   # vùng mặt
sky      = img_rgb[0:150,    0:70]     # vùng nền xanh
clothing = img_rgb[300:400, 100:300]   # vùng áo

# 2. Tính histogram màu 3D (8 bins/kênh → 512-D vector)
def color_hist(img, bins=8):
    h = cv2.calcHist([img], [0, 1, 2], None,
                     [bins, bins, bins],
                     [0, 256, 0, 256, 0, 256])
    h = h.flatten()
    return h / h.sum()      # chuẩn hóa thành phân bố xác suất

feat_face  = color_hist(face)
feat_sky   = color_hist(sky)
feat_cloth = color_hist(clothing)
print(f"Kích thước feature vector: {feat_face.shape[0]} chiều")

# 3. Khoảng cách giữa các feature
def dist(a, b):
    return np.linalg.norm(a - b)

print(f"d(face, sky)     = {dist(feat_face, feat_sky):.4f}")
print(f"d(face, clothing)= {dist(feat_face, feat_cloth):.4f}")
print(f"d(sky, clothing) = {dist(feat_sky, feat_cloth):.4f}")

# 4. Hiển thị
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (patch, name) in enumerate(zip([face, sky, clothing],
                                       ['Mặt', 'Nền xanh', 'Áo'])):
    axes[0, i].imshow(patch)
    axes[0, i].set_title(f'Vùng: {name} — shape {patch.shape}')
    axes[0, i].axis('off')

# Plot histogram R, G, B cho từng vùng
colors = ['red', 'green', 'blue']
for i, (patch, name) in enumerate(zip([face, sky, clothing],
                                       ['Mặt', 'Nền xanh', 'Áo'])):
    for ch, c in enumerate(colors):
        hist = cv2.calcHist([patch], [ch], None, [64], [0, 256]).flatten()
        axes[1, i].plot(hist, color=c, alpha=0.7)
    axes[1, i].set_title(f'Histogram RGB — {name}')
    axes[1, i].set_xlabel('Mức xám')

plt.suptitle('Bài 1: Đặc trưng ảnh — Histogram màu',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/bai1_color_histogram.png', dpi=100, bbox_inches='tight')
plt.show()

**📸 Kết quả:** Feature vector 512 chiều cho mỗi vùng; vùng "nền xanh" khác biệt rõ với "mặt" và "áo".

---

## Bài 2: Phân loại ảnh với đặc trưng + k-NN

**Mức độ:** Trung bình

**Mục tiêu:** Áp dụng pipeline **trích xuất đặc trưng → huấn luyện → dự đoán** cho bài toán phân loại ảnh.

**Yêu cầu:**
1. Tự sinh **dataset tổng hợp** gồm 3 lớp hình học: **tròn**, **vuông**, **tam giác** (mỗi lớp 30 ảnh 64×64).
2. Trích xuất đặc trưng mỗi ảnh: `[area, perimeter, số đỉnh, circularity, aspect_ratio]` từ contour.
3. Chia train/test 80/20.
4. Huấn luyện **k-NN** và **SVM** (scikit-learn).
5. In accuracy, confusion matrix, và dự đoán thử vài mẫu.
6. Lưu kết quả vào `output/bai2_classification.png`.

**Lời giải:**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# ---------- 1. Sinh dataset tổng hợp ----------
def gen_shape(shape_type, size=64, noise=0.0):
    """Sinh ảnh 1 hình trên nền đen."""
    img = np.zeros((size, size), dtype=np.uint8)
    center = (size // 2, size // 2)
    if shape_type == 'circle':
        cv2.circle(img, center, size // 3, 255, -1)
    elif shape_type == 'square':
        s = size // 3
        cv2.rectangle(img, (center[0] - s, center[1] - s),
                            (center[0] + s, center[1] + s), 255, -1)
    elif shape_type == 'triangle':
        s = size // 3
        pts = np.array([[center[0], center[1] - s],
                        [center[0] - s, center[1] + s],
                        [center[0] + s, center[1] + s]])
        cv2.fillPoly(img, [pts], 255)
    # Thêm nhiễu nhẹ
    if noise > 0:
        n = np.random.normal(0, noise * 255, img.shape)
        img = np.clip(img.astype(np.float32) + n, 0, 255).astype(np.uint8)
    return img


def extract_features(img):
    """Trích xuất đặc trưng hình học từ contour."""
    _, th = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(cnts) == 0:
        return np.zeros(5)
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    peri = cv2.arcLength(c, True)
    # Xấp xỉ đa giác để đếm số đỉnh
    approx = cv2.approxPolyDP(c, 0.02 * peri, True)
    n_vertices = len(approx)
    # Circularity
    circularity = 4 * np.pi * area / (peri ** 2) if peri > 0 else 0
    # Aspect ratio của bounding box
    x, y, w, h = cv2.boundingRect(c)
    aspect = w / h if h > 0 else 1
    return np.array([area, peri, n_vertices, circularity, aspect])


# Sinh dataset
np.random.seed(42)
X, y = [], []
label_map = {'circle': 0, 'square': 1, 'triangle': 2}
for label_name, label_id in label_map.items():
    for _ in range(30):
        img = gen_shape(label_name, noise=np.random.uniform(0, 0.05))
        X.append(extract_features(img))
        y.append(label_id)
X = np.array(X)
y = np.array(y)

# 2. Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                           random_state=42,
                                           stratify=y)
print(f"Train: {len(X_tr)} mẫu | Test: {len(X_te)} mẫu")

# 3. Huấn luyện 2 mô hình
knn = KNeighborsClassifier(n_neighbors=3).fit(X_tr, y_tr)
svm = SVC(kernel='rbf', C=1.0).fit(X_tr, y_tr)

for name, model in [('k-NN (k=3)', knn), ('SVM (RBF)', svm)]:
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    print(f"\n{name}: Accuracy = {acc*100:.2f}%")
    print("Confusion matrix:")
    print(confusion_matrix(y_te, y_pred))

# 4. Trực quan hóa
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# Vẽ 3 mẫu mỗi lớp
for i, (lname, lid) in enumerate(label_map.items()):
    imgs = [gen_shape(lname, noise=0.02) for _ in range(3)]
    combined = np.hstack(imgs)
    axes[i].imshow(combined, cmap='gray')
    axes[i].set_title(f'{lname} (label={lid})')
    axes[i].axis('off')

plt.suptitle('Bài 2: 3 lớp hình học — Tròn / Vuông / Tam giác',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/bai2_classification.png', dpi=100, bbox_inches='tight')
plt.show()

**📸 Kết quả:** Cả 2 mô hình đều đạt accuracy cao (>95%); số đỉnh (`n_vertices`) là đặc trưng quan trọng nhất.

---

## Bài 3: Phát hiện khuôn mặt với Haar Cascade

**Mức độ:** Trung bình

**Mục tiêu:** Áp dụng **Haar Cascade** có sẵn của OpenCV để phát hiện mặt người.

**Yêu cầu:**
1. Tải ảnh `data.astronaut()`.
2. Load `haarcascade_frontalface_default.xml`.
3. Phát hiện mặt với 2 cặp tham số khác nhau:
   - `scaleFactor=1.1, minNeighbors=5`
   - `scaleFactor=1.3, minNeighbors=3`
4. Vẽ bounding box lên ảnh gốc.
5. Hiển thị 3 ảnh (gốc + 2 kết quả).
6. Lưu kết quả vào `output/bai3_face_detect.png`.

**Lời giải:**

In [ ]:
img_rgb = data.astronaut()
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

# ✅ SỬA: Load Haar Cascade từ file đã tải trong thư mục images/
face_cascade = cv2.CascadeClassifier('images/haarcascade_frontalface_default.xml')

# Kiểm tra xem file đã load thành công chưa
if face_cascade.empty():
    raise IOError("Không thể tải file XML. Hãy kiểm tra lại đường dẫn và đảm bảo file đã được tải về đúng thư mục.")
else:
    print("✓ Đã tải thành công bộ phân loại Haar Cascade.")

# 2. Phát hiện với 2 cặp tham số
configs = [
    (1.1, 5, 'scaleFactor=1.1, minNeighbors=5'),
    (1.3, 3, 'scaleFactor=1.3, minNeighbors=3'),
]

vis_imgs = []
for sf, mn, title in configs:
    faces = face_cascade.detectMultiScale(img_gray,
                                           scaleFactor=sf,
                                           minNeighbors=mn,
                                           minSize=(30, 30))
    out = img_rgb.copy()
    for (x, y, w, h) in faces:
        cv2.rectangle(out, (x, y), (x + w, y + h), (255, 0, 0), 3)
    vis_imgs.append(out)
    print(f"{title}: phát hiện {len(faces)} khuôn mặt")

show_grid(
    [img_rgb] + vis_imgs,
    ['Ảnh gốc'] + [c[2] for c in configs],
    ncols=3, figsize=(16, 5),
    save_path='output/bai3_face_detect.png',
    main_title='Bài 3: Phát hiện khuôn mặt với Haar Cascade'
)

# NHẬN XÉT:
# - scaleFactor nhỏ + minNeighbors lớn → chính xác hơn nhưng có thể bỏ sót.
# - scaleFactor lớn + minNeighbors nhỏ → nhạy hơn nhưng dễ false positive.

**📸 Kết quả:** Haar Cascade phát hiện chính xác khuôn mặt phi hành gia; tham số ảnh hưởng đến độ nhạy.

---

## Bài 4: Phát hiện mắt và nụ cười

**Mức độ:** Trung bình

**Mục tiêu:** Kết hợp nhiều cascade để phát hiện chi tiết khuôn mặt.

**Yêu cầu:**
1. Tải ảnh `data.astronaut()`, chuyển xám.
2. Load 3 cascade: mặt, mắt, mỉm cười.
3. Phát hiện mặt trước, sau đó **chỉ tìm mắt/miệng trong vùng mặt** (tăng tốc + giảm false positive).
4. Vẽ bounding box màu khác nhau cho mặt / mắt / miệng.
5. Hiển thị và lưu vào `output/bai4_face_parts.png`.

**Lời giải:**

In [ ]:
img_rgb = data.astronaut()
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

# 1. Load 3 cascade + kiểm tra
face_cascade  = cv2.CascadeClassifier('images/haarcascade_frontalface_default.xml')
eye_cascade   = cv2.CascadeClassifier('images/haarcascade_eye.xml')
smile_cascade = cv2.CascadeClassifier('images/haarcascade_smile.xml')

# ✅ KIỂM TRA — phát hiện sớm file nào thiếu
for name, c in [('face', face_cascade),
                ('eye', eye_cascade),
                ('smile', smile_cascade)]:
    if c.empty():
        raise IOError(f"Cascade '{name}' rỗng → file XML chưa tải hoặc sai đường dẫn!")
    else:
        print(f"  ✓ Cascade '{name}' đã sẵn sàng")

# 2. Phát hiện mặt
faces = face_cascade.detectMultiScale(img_gray, 1.1, 5, minSize=(50, 50))
out = img_rgb.copy()

for (x, y, w, h) in faces:
    cv2.rectangle(out, (x, y), (x + w, y + h), (255, 0, 0), 3)

    roi_gray  = img_gray[y:y+h, x:x+w]
    roi_color = out[y:y+h, x:x+w]

    # 3. Tìm mắt trong nửa trên
    upper_half = roi_gray[:h//2, :]
    eyes = eye_cascade.detectMultiScale(upper_half, 1.1, 4, minSize=(15, 15))
    for (ex, ey, ew, eh) in eyes:
        cv2.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), (0, 255, 0), 2)

    # 4. Tìm miệng trong nửa dưới
    lower_half = roi_gray[h//2:, :]
    smiles = smile_cascade.detectMultiScale(lower_half, 1.7, 22, minSize=(25, 25))
    for (sx, sy, sw, sh) in smiles:
        cv2.rectangle(roi_color,
                      (sx, sy + h//2),
                      (sx + sw, sy + sh + h//2), (255, 255, 0), 2)

print(f"\nSố khuôn mặt phát hiện: {len(faces)}")

show_image(out, 'Bài 4: Phát hiện mặt / mắt / miệng',
           figsize=(7, 7),
           save_path='output/bai4_face_parts.png')

**📸 Kết quả:** Bounding box đỏ (mặt), xanh lá (mắt), vàng (miệng). Cascade hoạt động tốt hơn khi tìm trên vùng ROI nhỏ.

---

## Bài 5: Phát hiện điểm đặc trưng với ORB

**Mức độ:** Trung bình

**Mục tiêu:** Hiểu khái niệm **keypoint + descriptor** trong CV.

**Yêu cầu:**
1. Tải ảnh `data.astronaut()`, chuyển xám.
2. Tạo ORB detector với `nfeatures=200`.
3. Phát hiện keypoint + descriptor.
4. Vẽ keypoint lên ảnh (dùng `cv2.drawKeypoints`).
5. So sánh số keypoint với `nfeatures ∈ {50, 100, 500}`.
6. In shape của descriptor array.
7. Lưu kết quả vào `output/bai5_orb.png`.

**Lời giải:**

In [ ]:
img_rgb = data.astronaut()
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

# 1. ORB với các nfeatures khác nhau
nfeatures_list = [50, 100, 500]
results = []
counts  = []

for nf in nfeatures_list:
    orb = cv2.ORB_create(nfeatures=nf)
    kps, desc = orb.detectAndCompute(img_gray, None)
    counts.append(len(kps))
    out = cv2.drawKeypoints(img_rgb, kps, None,
                             color=(0, 255, 0),
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    results.append(out)

print(f"{'nfeatures':>12}{'keypoint thực tế':>20}{'descriptor shape':>20}")
print("-" * 52)
for nf in nfeatures_list:
    orb = cv2.ORB_create(nfeatures=nf)
    kps, desc = orb.detectAndCompute(img_gray, None)
    print(f"{nf:>12}{len(kps):>20}{str(desc.shape):>20}")

show_grid(
    [img_rgb] + results,
    ['Ảnh gốc'] + [f'ORB nfeatures={nf} → {cnt} kp'
                    for nf, cnt in zip(nfeatures_list, counts)],
    ncols=2, figsize=(13, 10),
    save_path='output/bai5_orb.png',
    main_title='Bài 5: Phát hiện điểm đặc trưng với ORB'
)

# NHẬN XÉT:
# - Descriptor ORB: 32 byte (256 bit) cho mỗi keypoint.
# - nfeatures lớn → nhiều keypoint hơn nhưng cũng có thể nhiều điểm yếu.

**📸 Kết quả:** Keypoint tập trung ở vùng có cấu trúc cao (mắt, tóc, chi tiết áo).

---

## Bài 6: Ghép ảnh panorama với ORB + BFMatcher

**Mức độ:** Nâng cao

**Mục tiêu:** Áp dụng keypoint + descriptor để **ghép 2 ảnh** (image stitching).

**Yêu cầu:**
1. Tạo 2 ảnh overlap bằng cách cắt từ `data.astronaut()` với offset khác nhau.
2. Phát hiện ORB keypoint trên cả 2 ảnh.
3. Match descriptor bằng `BFMatcher` + `crossCheck=True`.
4. Lọc match tốt bằng Lowe's ratio test (hoặc top-N match).
5. Tính `Homography` bằng `cv2.findHomography` (RANSAC).
6. Warp và ghép 2 ảnh lại.
7. Hiển thị 2 ảnh gốc + match + kết quả ghép; lưu vào `output/bai6_stitch.png`.

**Lời giải:**

In [ ]:
img_full = data.astronaut()
# Tạo 2 ảnh overlap (offset 150 pixel)
offset = 150
img1 = img_full[:, :350].copy()                # ảnh bên trái
img2 = img_full[:, offset:offset+350].copy()   # ảnh bên phải

# 1. ORB
orb = cv2.ORB_create(nfeatures=1000)
kps1, desc1 = orb.detectAndCompute(cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY), None)
kps2, desc2 = orb.detectAndCompute(cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY), None)
print(f"Ảnh 1: {len(kps1)} keypoint")
print(f"Ảnh 2: {len(kps2)} keypoint")

# 2. Matching
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(desc1, desc2)
matches = sorted(matches, key=lambda x: x.distance)   # sort theo khoảng cách
print(f"Tổng số match: {len(matches)}")

# 3. Trực quan hóa match (top 50)
match_vis = cv2.drawMatches(img1, kps1, img2, kps2,
                             matches[:50], None, flags=2)

# 4. Tính Homography (cần ít nhất 4 match)
if len(matches) >= 4:
    src_pts = np.float32([kps1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kps2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    print(f"Số inlier: {mask.sum()}/{len(mask)}")

    # 5. Warp và ghép
    h, w = img_full.shape[:2]
    result = cv2.warpPerspective(img1, H, (w + offset, h))
    result[:, offset:offset+350] = img2   # dán đè ảnh 2
else:
    result = np.hstack([img1, img2])
    mask = None

show_grid(
    [img1, img2, match_vis, result],
    ['Ảnh 1 (trái)', 'Ảnh 2 (phải)',
     f'Top-50 matches (inlier: {mask.sum() if mask is not None else 0})',
     'Kết quả ghép'],
    ncols=2, figsize=(14, 10),
    save_path='output/bai6_stitch.png',
    main_title='Bài 6: Ghép ảnh với ORB + BFMatcher + Homography'
)

cv2.imwrite('output/bai6_result.png', cv2.cvtColor(result, cv2.COLOR_RGB2BGR))

**📸 Kết quả:** Các đường match nối đúng điểm tương ứng giữa 2 ảnh; ảnh ghép mượt ở vùng overlap.

---

## Bài 7: Template Matching — Phát hiện đối tượng theo mẫu

**Mức độ:** Trung bình

**Mục tiêu:** Phát hiện đối tượng trong ảnh bằng cách tìm mẫu khớp.

**Yêu cầu:**
1. Tải ảnh `data.coins()`.
2. Cắt một đồng xu làm **template**.
3. Áp dụng `cv2.matchTemplate` với 3 phương pháp:
   - `TM_CCOEFF_NORMED`
   - `TM_CCORR_NORMED`
   - `TM_SQDIFF_NORMED`
4. Tìm vị trí khớp tốt nhất bằng `cv2.minMaxLoc`.
5. Vẽ bounding box kết quả.
6. Lưu kết quả vào `output/bai7_template.png`.

**Lời giải:**

In [ ]:
img = data.coins()

# 1. Chọn template (một đồng xu)
# Điều chỉnh tọa độ tùy theo ảnh
template = img[75:150, 230:305].copy()   # ví dụ: đồng xu góc trên-phải
h_t, w_t = template.shape

# 2. Áp dụng matchTemplate với 3 phương pháp
methods = {
    'TM_CCOEFF_NORMED': cv2.TM_CCOEFF_NORMED,
    'TM_CCORR_NORMED':  cv2.TM_CCORR_NORMED,
    'TM_SQDIFF_NORMED': cv2.TM_SQDIFF_NORMED,
}

results = []
for name, method in methods.items():
    res = cv2.matchTemplate(img, template, method)
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res)
    # Với SQDIFF, giá trị NHỎ là tốt; với 2 cái còn lại, giá trị LỚN là tốt
    if method == cv2.TM_SQDIFF_NORMED:
        top_left = min_loc
        score = min_val
    else:
        top_left = max_loc
        score = max_val
    bottom_right = (top_left[0] + w_t, top_left[1] + h_t)

    out = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    cv2.rectangle(out, top_left, bottom_right, (255, 0, 0), 2)
    results.append((out, f'{name}\nScore={score:.3f}'))
    print(f"{name}: score = {score:.4f}, vị trí = {top_left}")

# 3. Hiển thị
show_grid(
    [img, template] + [r[0] for r in results],
    ['Ảnh gốc', 'Template'] + [r[1] for r in results],
    ncols=3, figsize=(15, 10),
    save_path='output/bai7_template.png',
    main_title='Bài 7: Template Matching — phát hiện đồng xu'
)

# NHẬN XÉT:
# - Template matching tìm vị trí khớp mẫu tốt cho ảnh đơn giản.
# - Nhược: Không xử lý được xoay, scale, hoặc biến đổi phối cảnh.

**📸 Kết quả:** Cả 3 phương pháp tìm đúng đồng xu; CCOEFF thường ổn định nhất với ánh sáng.

---

## Bài 8: Đếm đối tượng trong ảnh

**Mức độ:** Trung bình

**Mục tiêu:** Pipeline **tiền xử lý → phân đoạn → contour → đếm** (một ứng dụng CV cổ điển).

**Yêu cầu:**
1. Tải ảnh `data.coins()`.
2. Pipeline:
   - Làm mịn Gaussian.
   - Otsu threshold.
   - Morphology (mở + đóng).
   - `findContours`.
   - Lọc theo diện tích (loại bỏ nhiễu nhỏ).
3. Vẽ contour + gán số thứ tự cho từng đối tượng.
4. In số đồng xu đếm được.
5. Lưu kết quả vào `output/bai8_counting.png`.

**Lời giải:**

In [ ]:
img = data.coins()

# 1. Pipeline
blur = cv2.GaussianBlur(img, (5, 5), 1.5)
_, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
opened = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=2)
closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=2)

# 2. Tìm contour
contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)

# 3. Lọc theo diện tích
min_area = 200
valid = [c for c in contours if cv2.contourArea(c) > min_area]
print(f"Tổng số contour: {len(contours)}")
print(f"Số đối tượng hợp lệ (>200px²): {len(valid)}")

# 4. Vẽ contour + số thứ tự
out = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
for i, c in enumerate(valid, 1):
    cv2.drawContours(out, [c], -1, (0, 255, 0), 2)
    # Tâm contour
    M = cv2.moments(c)
    if M['m00'] > 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        cv2.putText(out, str(i), (cx - 8, cy + 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

show_grid(
    [img, th, closed, out],
    ['Ảnh gốc', 'Otsu', 'Sau Morphology', f'Đếm được {len(valid)} đồng xu'],
    ncols=4, figsize=(18, 5),
    save_path='output/bai8_counting.png',
    main_title='Bài 8: Đếm đối tượng — pipeline đếm đồng xu'
)

**📸 Kết quả:** Đếm chính xác số đồng xu, loại bỏ nhiễu nhỏ bằng lọc diện tích.

---

## Bài 9: Đo lường kích thước đối tượng

**Mức độ:** Trung bình

**Mục tiêu:** Từ contour, đo diện tích, chu vi và ước lượng đường kính đối tượng.

**Yêu cầu:**
1. Tải ảnh `data.coins()` (đã biết tỷ lệ: 1 pixel ≈ 0.1 mm — giả định).
2. Tìm contour (như Bài 8).
3. Với mỗi đối tượng, tính:
   - Diện tích (pixel²)
   - Chu vi (pixel)
   - **Đường kính tương đương** (từ diện tích: `d = 2·√(A/π)`)
   - **Circularity** = `4πA / P²` (đo độ tròn: 1.0 = tròn hoàn hảo)
4. In bảng kết quả cho 5 đồng xu đầu.
5. Vẽ bounding box + chú thích đường kính; lưu vào `output/bai9_measure.png`.

**Lời giải:**

In [ ]:
img = data.coins()
PIXEL_TO_MM = 0.1    # giả định tỷ lệ

# Pipeline phân đoạn
blur = cv2.GaussianBlur(img, (5, 5), 1.5)
_, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
closed = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=2)
contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
valid = sorted([c for c in contours if cv2.contourArea(c) > 200],
               key=cv2.contourArea, reverse=True)

# Đo lường
print(f"{'#':>3}{'Diện tích':>12}{'Chu vi':>10}{'Đ.kính (px)':>14}"
      f"{'Đ.kính (mm)':>14}{'Tròn?':>8}")
print("-" * 63)

out = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
measurements = []
for i, c in enumerate(valid, 1):
    area = cv2.contourArea(c)
    peri = cv2.arcLength(c, True)
    diameter_px = 2 * np.sqrt(area / np.pi)
    diameter_mm = diameter_px * PIXEL_TO_MM
    circularity = 4 * np.pi * area / (peri ** 2) if peri > 0 else 0

    # Vẽ
    (x, y), r = cv2.minEnclosingCircle(c)
    cv2.circle(out, (int(x), int(y)), int(r), (0, 255, 0), 2)
    cv2.putText(out, f'{diameter_mm:.1f}mm',
                (int(x) - 30, int(y) + 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

    measurements.append((i, area, peri, diameter_px, diameter_mm, circularity))
    if i <= 5:
        print(f"{i:>3}{area:>12.0f}{peri:>10.1f}{diameter_px:>14.1f}"
              f"{diameter_mm:>14.1f}{circularity:>8.3f}")

show_image(out, f'Bài 9: Đo lường {len(valid)} đồng xu',
           figsize=(7, 7),
           save_path='output/bai9_measure.png')

**📸 Kết quả:** Bảng đo đường kính từng đồng xu; circularity > 0.9 chứng tỏ hình gần tròn.

---

## Bài 10: Phân đoạn ngữ nghĩa đơn giản (Semantic Segmentation)

**Mức độ:** Trung bình

**Mục tiêu:** Gán nhãn **lớp** cho từng pixel dựa trên ngưỡng (minh họa semantic segmentation).

**Yêu cầu:**
1. Tải ảnh `data.camera()`.
2. Xác định 3 lớp pixel bằng ngưỡng thủ công:
   - **Tối** (nền): `I < 80`
   - **Trung bình** (da/áo): `80 ≤ I < 160`
   - **Sáng** (mặt/chi tiết): `I ≥ 160`
3. Tạo ảnh phân đoạn với **color map** (ví dụ: đen / xanh lá / vàng).
4. Overlay lên ảnh gốc với alpha = 0.5.
5. In **tỷ lệ pixel** của mỗi lớp.
6. Lưu kết quả vào `output/bai10_semantic.png`.

**Lời giải:**

In [ ]:
img = data.camera()

# 1. Phân lớp theo ngưỡng
label_map = np.zeros_like(img, dtype=np.uint8)
label_map[img < 80]  = 0    # nền tối
label_map[(img >= 80) & (img < 160)] = 1   # trung bình
label_map[img >= 160] = 2   # sáng

# 2. Color map
palette = np.array([
    [  0,   0,   0],      # đen — lớp 0
    [ 50, 200,  50],      # xanh lá — lớp 1
    [255, 220,   0],      # vàng — lớp 2
], dtype=np.uint8)
seg_color = palette[label_map]

# 3. Overlay
img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
overlay = (0.6 * img_rgb + 0.4 * seg_color).astype(np.uint8)

# 4. Tỷ lệ pixel mỗi lớp
total = img.size
for lbl, name in [(0, 'Tối (nền)'), (1, 'Trung bình'), (2, 'Sáng')]:
    ratio = np.sum(label_map == lbl) / total * 100
    print(f"Lớp {lbl} - {name}: {ratio:.1f}%")

show_grid(
    [img, seg_color, overlay],
    ['Ảnh gốc (xám)',
     'Phân đoạn ngữ nghĩa\n(0=đen, 1=xanh, 2=vàng)',
     'Overlay (alpha=0.4)'],
    ncols=3, figsize=(15, 5),
    save_path='output/bai10_semantic.png',
    main_title='Bài 10: Phân đoạn ngữ nghĩa đơn giản bằng ngưỡng'
)

**📸 Kết quả:** Ảnh được gán nhãn theo 3 mức sáng; overlay giúp thấy rõ phân bố các lớp.

---

## Bài 11: Trích xuất đặc trưng HOG + SVM

**Mức độ:** Nâng cao

**Mục tiêu:** Áp dụng **HOG (Histogram of Oriented Gradients)** + **SVM** cho phân loại ảnh.

**Yêu cầu:**
1. Sinh dataset tổng hợp gồm **đường ngang** và **đường dọc** (mỗi lớp 40 ảnh 64×64).
2. Trích xuất HOG descriptor cho mỗi ảnh.
3. Huấn luyện SVM (linear kernel) trên feature HOG.
4. Đánh giá accuracy trên tập test 20%.
5. Trực quan hóa HOG của một mẫu mỗi lớp.
6. Lưu kết quả vào `output/bai11_hog.png`.

**Lời giải:**

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from skimage.feature import hog

# 1. Sinh dataset
def gen_line(orientation, size=64, noise=0.05):
    img = np.zeros((size, size), dtype=np.uint8)
    if orientation == 'horizontal':
        for y in [size//2 - 1, size//2, size//2 + 1]:
            img[y, 10:size-10] = 255
    else:
        for x in [size//2 - 1, size//2, size//2 + 1]:
            img[10:size-10, x] = 255
    n = np.random.normal(0, noise * 255, img.shape)
    return np.clip(img.astype(np.float32) + n, 0, 255).astype(np.uint8)

def extract_hog(img):
    return hog(img, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), feature_vector=True)

X, y = [], []
for _ in range(40):
    X.append(extract_hog(gen_line('horizontal')));  y.append(0)
    X.append(extract_hog(gen_line('vertical')));    y.append(1)
X = np.array(X); y = np.array(y)
print(f"Feature HOG shape: {X.shape}")

# 2. Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                           random_state=42, stratify=y)

# 3. Huấn luyện SVM
svm = SVC(kernel='linear', C=1.0).fit(X_tr, y_tr)
y_pred = svm.predict(X_te)
acc = accuracy_score(y_te, y_pred)
print(f"SVM + HOG — Accuracy: {acc*100:.2f}%")

# 4. Trực quan hóa HOG
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for i, (orient, name) in enumerate([('horizontal', 'Ngang'),
                                     ('vertical', 'Dọc')]):
    img = gen_line(orient, noise=0.02)
    _, hog_img = hog(img, orientations=9, pixels_per_cell=(8, 8),
                     cells_per_block=(2, 2), visualize=True)
    axes[i, 0].imshow(img, cmap='gray'); axes[i, 0].set_title(f'{name} — Ảnh gốc'); axes[i, 0].axis('off')
    axes[i, 1].imshow(hog_img, cmap='gray'); axes[i, 1].set_title(f'{name} — HOG'); axes[i, 1].axis('off')

plt.suptitle('Bài 11: HOG + SVM phân loại đường ngang/dọc',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/bai11_hog.png', dpi=100, bbox_inches='tight')
plt.show()

# NHẬN XÉT:
# - HOG mô tả phân bố hướng gradient trong ô cục bộ.
# - Với đường ngang/dọc, HOG cho feature phân biệt hoàn hảo.

**📸 Kết quả:** Accuracy 100% trên bài toán đơn giản; HOG visual rõ sự khác biệt hướng gradient.

---

## Bài 12: OCR đơn giản với Template Matching

**Mức độ:** Nâng cao

**Mục tiêu:** Áp dụng template matching để đọc chữ số (minh họa OCR cổ điển).

**Yêu cầu:**
1. Sinh 10 ảnh chữ số `0-9` (dùng `cv2.putText`) làm **template**.
2. Sinh ảnh "biển số" tổng hợp chứa chuỗi số (ví dụ "2024") với kích thước khác.
3. Với mỗi vị trí trong ảnh biển số, so khớp với 10 template, chọn template có score cao nhất.
4. Vẽ bounding box + nhãn dự đoán.
5. In chuỗi đọc được so với ground truth.
6. Lưu kết quả vào `output/bai12_ocr.png`.

**Lời giải:**

In [ ]:
# 1. Tạo template chữ số
def make_digit(d, size=(60, 40)):
    img = np.zeros(size, dtype=np.uint8)
    cv2.putText(img, str(d), (5, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, 255, 3)
    return img

templates = {d: make_digit(d) for d in range(10)}

# 2. Tạo ảnh "biển số"
plate = np.zeros((80, 320), dtype=np.uint8)
cv2.putText(plate, "2024", (10, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 2.0, 255, 4)
plate = cv2.copyMakeBorder(plate, 10, 10, 10, 10,
                            cv2.BORDER_CONSTANT, value=0)

# 3. Sliding window matching
h_t, w_t = templates[0].shape
best_matches = []
# Chia ảnh thành 4 ô đều để demo
step = plate.shape[1] // 4
for i in range(4):
    x0 = i * step
    x1 = x0 + step
    roi = plate[:, x0:x1]

    # Resize roi bằng kích thước template
    roi_rs = cv2.resize(roi, (w_t, h_t))
    best_digit, best_score = None, -np.inf
    for d, tmpl in templates.items():
        res = cv2.matchTemplate(roi_rs, tmpl, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, _ = cv2.minMaxLoc(res)
        if max_val > best_score:
            best_score = max_val
            best_digit = d
    best_matches.append((best_digit, best_score, x0, x1))

# 4. Vẽ kết quả
out = cv2.cvtColor(plate, cv2.COLOR_GRAY2RGB)
predicted = ""
for digit, score, x0, x1 in best_matches:
    predicted += str(digit)
    cv2.rectangle(out, (x0, 0), (x1, out.shape[0]), (0, 255, 0), 1)
    cv2.putText(out, str(digit), (x0 + 10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

ground_truth = "2024"
print(f"Ground truth: {ground_truth}")
print(f"Dự đoán     : {predicted}")
print(f"Khớp?       : {predicted == ground_truth}")

show_image(out, f'Bài 12: OCR đơn giản\nGT="{ground_truth}"  |  Dự đoán="{predicted}"',
           figsize=(10, 3),
           save_path='output/bai12_ocr.png')

**📸 Kết quả:** Đọc được chuỗi số 2024 (minh họa nguyên lý OCR cổ điển).

---

## Bài 13: So sánh các bộ phát hiện keypoint — ORB vs SIFT vs AKAZE

**Mức độ:** Nâng cao

**Mục tiêu:** Hiểu sự khác biệt giữa các **feature detector** phổ biến.

**Yêu cầu:**
1. Tải ảnh `data.camera()`.
2. Chạy 3 detector: `ORB`, `SIFT`, `AKAZE`.
3. Đo:
   - Số keypoint
   - Thời gian phát hiện (ms)
   - Kích thước descriptor (byte)
4. Vẽ keypoint trên ảnh.
5. In bảng so sánh.
6. Lưu kết quả vào `output/bai13_detectors.png`.

**Lời giải:**

In [ ]:
import time

img = data.camera()

# 1. Chuẩn bị 2 detector có sẵn trong opencv-python tiêu chuẩn
detectors = {
    'ORB':  cv2.ORB_create(nfeatures=500),
    'SIFT': cv2.SIFT_create(nfeatures=500),
}

# 2. Chạy và đo
results = []
print(f"{'Detector':<10}{'Số keypoint':>14}{'Thời gian (ms)':>18}{'Descriptor (byte)':>20}")
print("-" * 62)

for name, det in detectors.items():
    t0 = time.time()
    kps, desc = det.detectAndCompute(img, None)
    t1 = time.time()
    dt_ms = (t1 - t0) * 1000
    desc_bytes = desc.itemsize * desc.shape[1] if desc is not None else 0

    out = cv2.drawKeypoints(img, kps, None,
                            color=(0, 255, 0),
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    results.append((out, name, len(kps), dt_ms, desc_bytes))
    print(f"{name:<10}{len(kps):>14}{dt_ms:>18.2f}{desc_bytes:>20}")

# 3. Hiển thị
show_grid(
    [img] + [r[0] for r in results],
    ['Ảnh gốc'] + [f'{r[1]}\n{r[2]} kp — {r[3]:.1f} ms' for r in results],
    ncols=2, figsize=(14, 10),
    save_path='output/bai13_detectors.png',
    main_title='Bài 13: So sánh ORB / SIFT'
)

# NHẬN XÉT:
# - SIFT: chậm hơn nhưng chất lượng keypoint tốt, bất biến scale + rotation.
# - ORB : nhanh nhất, descriptor 32 byte, phù hợp real-time.

**📸 Kết quả:** Bảng so sánh cho thấy trade-off giữa tốc độ và chất lượng keypoint.

---

## Bài 14: Pipeline CV hoàn chỉnh — Đếm và phân loại đối tượng

**Mức độ:** Nâng cao

**Mục tiêu:** Tổng hợp toàn bộ Chương 1 → 5 vào một pipeline CV hoàn chỉnh.

**Yêu cầu:**
1. Tải ảnh `data.coins()`.
2. Pipeline:
   - **(Ch1)** Xem thông tin ảnh.
   - **(Ch2)** Làm mịn Gaussian.
   - **(Ch2)** Otsu threshold.
   - **(Ch4)** Morphology làm sạch + `findContours`.
   - **(Ch5)** Với mỗi contour, trích xuất đặc trưng: `[area, circularity, aspect]`.
   - **(Ch5)** Phân loại bằng rule-based (đơn giản) thành 3 nhóm: **nhỏ**, **vừa**, **lớn**.
3. Vẽ contour theo màu tương ứng với từng nhóm.
4. In bảng tổng hợp số lượng mỗi nhóm.
5. Lưu kết quả vào `output/bai14_pipeline.png`.

**Lời giải:**

In [ ]:
img = data.coins()

# 1. Tiền xử lý
blur = cv2.GaussianBlur(img, (5, 5), 1.5)
_, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 2. Morphology + contours
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
closed = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=2)
contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
valid = [c for c in contours if cv2.contourArea(c) > 200]

# 3. Trích xuất đặc trưng
feats = []
for c in valid:
    area = cv2.contourArea(c)
    peri = cv2.arcLength(c, True)
    circ = 4 * np.pi * area / (peri ** 2) if peri > 0 else 0
    x, y, w, h = cv2.boundingRect(c)
    aspect = w / h if h > 0 else 1
    feats.append({'contour': c, 'area': area, 'circularity': circ,
                  'aspect': aspect})

# 4. Phân loại rule-based theo diện tích
area_vals = np.array([f['area'] for f in feats])
thresh_small = np.percentile(area_vals, 33)
thresh_large = np.percentile(area_vals, 66)

def classify(f):
    if f['area'] < thresh_small:   return 'small'
    if f['area'] < thresh_large:   return 'medium'
    return 'large'

for f in feats:
    f['class'] = classify(f)

counts = Counter(f['class'] for f in feats)
print(f"Ngưỡng nhỏ/vừa: {thresh_small:.0f} / {thresh_large:.0f}")
print(f"Số đối tượng: {counts}")

# 5. Vẽ theo nhóm màu
color_map = {'small':  (255, 100, 100),   # xanh dương nhạt
             'medium': (100, 255, 100),   # xanh lá
             'large':  (255, 200,   0)}   # vàng
out = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
for f in feats:
    c = color_map[f['class']]
    cv2.drawContours(out, [f['contour']], -1, c, 2)
    M = cv2.moments(f['contour'])
    if M['m00'] > 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        cv2.putText(out, f['class'][0].upper(),
                    (cx - 6, cy + 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

# 6. Hiển thị
show_grid(
    [img, blur, th, closed, out],
    ['1. Ảnh gốc',
     '2. Gaussian Blur',
     '3. Otsu Threshold',
     '4. Morphology',
     f'5. Contour + Classify\nS={counts["small"]}, M={counts["medium"]}, L={counts["large"]}'],
    ncols=3, figsize=(15, 10),
    save_path='output/bai14_pipeline.png',
    main_title='Bài 14: Pipeline CV hoàn chỉnh — Phân đoạn + Trích xuất + Phân loại'
)

cv2.imwrite('output/bai14_result.png', cv2.cvtColor(out, cv2.COLOR_RGB2BGR))

**📸 Kết quả:** Pipeline phân loại đồng xu thành 3 nhóm kích thước, minh họa toàn bộ kiến thức từ Chương 1 → 5.

---

## 📌 Tổng kết kiến thức Chương 1 → 5

| Bài | Chương | Kiến thức chính |
|-----|--------|-----------------|
| 1 | 5 | Feature vector (color histogram) |
| 2 | 5 | Phân loại ảnh: đặc trưng + k-NN/SVM |
| 3 | 5 | Haar Cascade — phát hiện khuôn mặt |
| 4 | 5 | Nhiều cascade kết hợp (mặt/mắt/miệng) |
| 5 | 5 | Keypoint + descriptor (ORB) |
| 6 | 5 | Image Stitching: ORB + BFMatcher + Homography |
| 7 | 5 | Template Matching |
| 8 | 1, 4, 5 | Đếm đối tượng (Blur + Otsu + Contour) |
| 9 | 4, 5 | Đo lường kích thước đối tượng |
| 10 | 4, 5 | Semantic Segmentation bằng ngưỡng |
| 11 | 5 | HOG + SVM |
| 12 | 5 | OCR đơn giản (template matching) |
| 13 | 5 | So sánh ORB / SIFT / AKAZE |
| 14 | 1 → 5 | Pipeline CV hoàn chỉnh |

**✅ Đặc điểm:**
- Mỗi bài đều có **hiển thị ảnh minh họa** để sinh viên dễ theo dõi.
- Tất cả kết quả đều **tự động lưu** vào `output/`.
- Sử dụng **kiến thức Chương 1 → 5** — không dùng Deep Learning (TensorFlow/PyTorch) vì cần GPU và dataset lớn.
- Kết hợp nhiều chương ở các bài cuối (Bài 8, 9, 10, 14) để sinh viên thấy tính liên kết.

**💡 Ghi chú về Deep Learning (Chương 5 có đề cập):**
Chương 5 giới thiệu các mô hình **CNN (AlexNet, ResNet, YOLO, U-Net...)** nhưng để **chạy được** cần:
- Cài thêm `tensorflow` hoặc `torch` (vài GB).
- Dataset lớn có nhãn (ImageNet, COCO...).
- GPU (khuyến nghị) để huấn luyện.

Vì vậy bộ bài tập này **tập trung vào CV cổ điển** (Haar, HOG, ORB, SIFT, SVM, k-NN) — đúng với mục tiêu môn học và chạy được ngay trên CPU/notebook thông thường. Nếu bạn muốn bổ sung **1–2 bài về suy luận mô hình pretrained** (ví dụ: dùng `cv2.dnn.readNet` để chạy YOLO pretrained trên ảnh mẫu), tôi có thể thêm.

Bạn có muốn tôi bổ sung thêm bài về **OpenCV DNN + model pretrained** hoặc điều chỉnh gì không? 🚀